# 05 — CKD Modeling
**Objective:** Train, compare, calibrate and evaluate CKD disease models with strict data leakage prevention.

In [ ]:
import sys; sys.path.insert(0,'../ml/pipelines')
from train_models import load_ckd, candidates, build_preprocessor, evaluate
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
X, y = load_ckd()
print(f"Shape: {X.shape} | Positive class: {y.mean():.1%}")

## Train/Val/Test Split
All preprocessing is fitted ONLY on training data — no leakage.

In [ ]:
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=SEED)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

prep = build_preprocessor(X_train)
Xt = prep.fit_transform(X_train)  # FIT only on train
Xv = prep.transform(X_val)
Xts = prep.transform(X_test)

## Model Comparison

In [ ]:
results = {}
for name, clf in candidates().items():
    try:
        clf.fit(Xt, y_train)
        auc = roc_auc_score(y_val, clf.predict_proba(Xv)[:,1])
        results[name] = {'model': clf, 'val_auc': auc}
        print(f"  {name:30s}  val AUC = {auc:.4f}")
    except Exception as e:
        print(f"  {name} failed: {e}")

best_name = max(results, key=lambda k: results[k]['val_auc'])
best_model = results[best_name]['model']
print(f"\n→ Best: {best_name}")

## Calibration

In [ ]:
calibrated = CalibratedClassifierCV(best_model, cv='prefit', method='sigmoid')
calibrated.fit(Xv, y_val)
metrics = evaluate(calibrated, Xts, y_test)
print("\nTest Metrics:")
for k, v in metrics.items(): print(f"  {k:15s}: {v}")

## ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
RocCurveDisplay.from_estimator(calibrated, Xts, y_test, ax=ax, name='CKD Model')
ax.plot([0,1],[0,1],'k--',alpha=0.4)
ax.set_title('CKD — ROC Curve (Test Set)')
plt.tight_layout(); plt.show()

## Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(calibrated, Xts, y_test, ax=ax,
    display_labels=['Not CKD','CKD'], cmap='Blues')
ax.set_title('CKD — Confusion Matrix (Test Set)')
plt.tight_layout(); plt.show()

## Conclusions
Document selected model, metrics, and any observations about false negatives (critical for screening).